In [2]:
from dataGeneration import run_scr_model
from minMaxLHSampler import maximin_lhs

import numpy as np

mat_path = "./NEDC_lite.mat"

In [3]:
mean_k_ads = 5
mean_k_des = 5e04
mean_k_std = 4e09
mean_k_fst = 2e09
mean_k_slw = 6e11
mean_k_oxi = 7e03

one_std_k_ads = 2
one_std_k_des = 2e04
one_std_k_std = 8e08
one_std_k_fst = 5e08
one_std_k_slw = 2e11
one_std_k_oxi = 2e03

In [4]:
low_k_ads = mean_k_ads - 2*one_std_k_ads
up_k_ads  = mean_k_ads + 2*one_std_k_ads

low_k_des = mean_k_des - 2*one_std_k_des
up_k_des  = mean_k_des + 2*one_std_k_des

low_k_std = mean_k_std - 2*one_std_k_std
up_k_std  = mean_k_std + 2*one_std_k_std

low_k_fst = mean_k_fst - 2*one_std_k_fst
up_k_fst  = mean_k_fst + 2*one_std_k_fst

low_k_slw = mean_k_slw - 2*one_std_k_slw
up_k_slw  = mean_k_slw + 2*one_std_k_slw

low_k_oxi = mean_k_oxi - 2*one_std_k_oxi
up_k_oxi  = mean_k_oxi + 2*one_std_k_oxi

lower_bounds = [low_k_ads, low_k_des, low_k_std, low_k_fst, low_k_slw, low_k_oxi]
upper_bounds = [up_k_ads, up_k_des, up_k_std, up_k_fst, up_k_slw, up_k_oxi]

# ads = 0
# des = 1
# std = 2
# fst = 3
# slw = 4
# oxi = 5


In [5]:
import pandas as pd

bounds_df = pd.DataFrame({
    "parameter": ["k_ads", "k_des", "k_std", "k_fst", "k_slw", "k_oxi"],
    "low": lower_bounds,
    "up": upper_bounds,
})

bounds_df

,parameter,low,up
0,k_ads,1.000000e+00,9.000000e+00
1,k_des,1.000000e+04,9.000000e+04
2,k_std,2.400000e+09,5.600000e+09
3,k_fst,1.000000e+09,3.000000e+09
4,k_slw,2.000000e+11,1.000000e+12
5,k_oxi,3.000000e+03,1.100000e+04


In [21]:
best_sample = maximin_lhs(lower_bounds=lower_bounds, upper_bounds=upper_bounds, n_samples=5000, n_candidates=1000)

In [22]:
np.save("lhs_sampling.npy", best_sample)
print(best_sample.shape)

(5000, 6)


In [16]:
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# ------------------------------------------------------------
# Parallel forward simulations: 7 workers
# ------------------------------------------------------------

n_samples = len(best_sample)
max_workers = 7
save_every = 100
out_path = "lhs_data.npy"


def run_one_sample(i):
    k_ads = best_sample[i][0]
    k_des = best_sample[i][1]
    k_std = best_sample[i][2]
    k_fst = best_sample[i][3]
    k_slw = best_sample[i][4]
    k_oxi = best_sample[i][5]

    time_s, Model_sensor_ppm = run_scr_model(
        mat_path=mat_path,
        k_std_0=k_std,
        k_fst_0=k_fst,
        k_slw_0=k_slw,
        k_ads_0=k_ads,
        k_des_0=k_des,
        k_nh3ox_0=k_oxi,
    )

    return i, Model_sensor_ppm


data = None
n_done = 0

with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(run_one_sample, i) for i in range(n_samples)]

    for future in tqdm(
        as_completed(futures),
        total=n_samples,
        desc="Running SCR simulations",
    ):
        i, Model_sensor_ppm = future.result()

        if data is None:
            n_time = len(Model_sensor_ppm)
            data = np.zeros((n_samples, n_time))

        data[i, :] = Model_sensor_ppm
        n_done += 1

        if n_done % save_every == 0:
            np.save(out_path, data)
            print(f"Saved data for {n_done} completed samples.")

np.save(out_path, data)

print("Done.")
print(f"Saved {out_path}")
print("data shape:", data.shape)

Running SCR simulations: 100%|██████████████████| 10/10 [01:41<00:00, 10.11s/it]

Done.
Saved lhs_data.npy
data shape: (10, 11993)


In [24]:
outputs = np.load("lhs_sampling.npy")


In [25]:
outputs.shape

(5000, 6)

In [20]:
outputs[1]

array([ 0.        , 20.98308105, 30.0653182 , ...,  0.67025123,
        0.63004636,  0.6428641 ], shape=(11993,))

## Splitting data into Train, Val, Test

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split

outputs = np.load("../Data/lhs_data.npy")
ks = np.load("../Data/lhs_sampling.npy")

outputs_train, outputs_test, ks_train, ks_test = train_test_split(outputs, ks, test_size=0.2, random_state=0) # 80% train, 20% val+test
outputs_val, outputs_test, ks_val, ks_test = train_test_split(outputs_test, ks_test, test_size=0.5, random_state=0) # 10% val, 10% test 

print(outputs_train.shape, outputs_val.shape, outputs_test.shape)
print(ks_train.shape, ks_val.shape, ks_test.shape)


(800, 11993) (100, 11993) (100, 11993)
(800, 6) (100, 6) (100, 6)


In [2]:
np.save("../Data/outputs_train.npy", outputs_train)
np.save("../Data/outputs_val.npy", outputs_val)
np.save("../Data/outputs_test.npy", outputs_test)
np.save("../Data/ks_train.npy", ks_train)
np.save("../Data/ks_val.npy", ks_val)
np.save("../Data/ks_test.npy", ks_test)